# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [2]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [3]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [4]:
# links = fetch_website_links("https://edwarddonner.com")
links = fetch_website_links("https://codilime.com/")
links

['/',
 '/services/networks/',
 '/services/equipment/',
 '/services/environment/',
 '/services/data/',
 '/services/security/',
 '/services/product-design/',
 '/services/frontend-development/',
 '/services/backend-development/',
 '/services/low-level-programming-engineering/',
 '/services/devops/',
 '/services/platform-engineering-services/',
 '/services/test-automation/',
 '/services/embedded-software-development/',
 '/services/network-professional-services/',
 '/services/end-to-end-monitoring/',
 '/services/network-automation/',
 '/services/network-testing-services/',
 '/services/network-infrastructure-design/',
 '/services/data-engineering/',
 '/services/data-science/',
 '/services/rd-services/',
 '/services/mvp-development/',
 '/services/software-development-for-startups/',
 '/technology-stack/',
 '/case-studies/',
 '/blog/',
 '/services/networks/',
 '/services/equipment/',
 '/services/environment/',
 '/services/data/',
 '/services/security/',
 '/blog/',
 '/resources/',
 'https://res

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [8]:
# print(get_links_user_prompt("https://edwarddonner.com"))
print(get_links_user_prompt("https://codilime.com/"))


Here is the list of links on the website https://codilime.com/ -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

/
/services/networks/
/services/equipment/
/services/environment/
/services/data/
/services/security/
/services/product-design/
/services/frontend-development/
/services/backend-development/
/services/low-level-programming-engineering/
/services/devops/
/services/platform-engineering-services/
/services/test-automation/
/services/embedded-software-development/
/services/network-professional-services/
/services/end-to-end-monitoring/
/services/network-automation/
/services/network-testing-services/
/services/network-infrastructure-design/
/services/data-engineering/
/services/data-science/
/services/rd-services/
/services/mvp-development/
/services/software-development-for-startups/
/technology-s

In [12]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [11]:
select_relevant_links("https://codilime.com/")
# select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page', 'url': 'https://codilime.com/about/'},
  {'type': 'careers page', 'url': 'https://codilime.com/careers/'},
  {'type': 'case studies page', 'url': 'https://codilime.com/case-studies/'},
  {'type': 'technology stack page',
   'url': 'https://codilime.com/technology-stack/'},
  {'type': 'blog page', 'url': 'https://codilime.com/blog/'},
  {'type': 'news page', 'url': 'https://codilime.com/news/'},
  {'type': 'contact page', 'url': 'https://codilime.com/contact/'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/codilime'},
  {'type': 'Facebook page', 'url': 'https://www.facebook.com/codilime/'},
  {'type': 'X (Twitter) page', 'url': 'https://x.com/codilime'},
  {'type': 'Instagram profile', 'url': 'https://www.instagram.com/codilime/'},
  {'type': 'Dribbble profile', 'url': 'https://dribbble.com/CodiLime'},
  {'type': 'Behance profile', 'url': 'https://www.behance.net/codilime'},
  {'type': 'YouTube channel', 'url': 'https://www.yo

In [13]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [14]:
select_relevant_links("https://codilime.com/")

Selecting relevant links for https://codilime.com/ by calling gpt-5-nano
Found 27 relevant links


{'links': [{'type': 'about page', 'url': 'https://codilime.com/about/'},
  {'type': 'careers page', 'url': 'https://codilime.com/careers/'},
  {'type': 'case studies page', 'url': 'https://codilime.com/case-studies/'},
  {'type': 'blog', 'url': 'https://codilime.com/blog/'},
  {'type': 'news page', 'url': 'https://codilime.com/news/'},
  {'type': 'resources page', 'url': 'https://codilime.com/resources/'},
  {'type': 'technology stack page',
   'url': 'https://codilime.com/technology-stack/'},
  {'type': 'product design service page',
   'url': 'https://codilime.com/services/product-design/'},
  {'type': 'frontend development service page',
   'url': 'https://codilime.com/services/frontend-development/'},
  {'type': 'backend development service page',
   'url': 'https://codilime.com/services/backend-development/'},
  {'type': 'devops service page',
   'url': 'https://codilime.com/services/devops/'},
  {'type': 'data engineering service page',
   'url': 'https://codilime.com/services/da

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [15]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [17]:
# print(fetch_page_and_all_relevant_links("https://huggingface.co"))
print(fetch_page_and_all_relevant_links("https://codilime.com/"))


Selecting relevant links for https://codilime.com/ by calling gpt-5-nano
Found 41 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


## Landing Page:

CodiLime | Networking Expert Company & Strategic Partner

Services
Networks
Equipment
Environment
Data
Security
N
E
E
D
S
Services
DESIGN
Product design
SOFTWARE ENGINEERING
Frontend
Backend
Low-level engineering
DevOps
Platform engineering
Test automation
Embedded systems
NETWORK & CLOUD ENGINEERING
Network professional services
E2E monitoring & observability
Network automation
Network testing
Network infrastructure design
DATA
Data engineering
Data science
RESEARCH AND DEVELOPMENT
R&D
FOR STARTUPS
MVP software development
Software development for startups
Technologies
Case studies
Blog
Knowledge
Networks
Equipment
Environment
Data
Security
N
E
E
D
S
Knowledge
EXPERTISE
Blog
Resources
PUBLICATIONS
AI/ML for networks
Exploring SONiC's True Potential
Application Networking in Kubernetes
Hardware TCP Offloading
AI-based Web Application Firewall
NEWS
Newsroom
Newsletter
About
Careers
Contact Us
Networks
Equipment
Environment
Data
Security
N
E
E
D
S
Delivery excellence fo

In [18]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [19]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [20]:
# get_brochure_user_prompt("HuggingFace", "https://huggingface.co")
get_brochure_user_prompt("CodiLime", "https://codilime.com/")

Selecting relevant links for https://codilime.com/ by calling gpt-5-nano
Found 26 relevant links


"\nYou are looking at a company called: CodiLime\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nCodiLime | Networking Expert Company & Strategic Partner\n\nServices\nNetworks\nEquipment\nEnvironment\nData\nSecurity\nN\nE\nE\nD\nS\nServices\nDESIGN\nProduct design\nSOFTWARE ENGINEERING\nFrontend\nBackend\nLow-level engineering\nDevOps\nPlatform engineering\nTest automation\nEmbedded systems\nNETWORK & CLOUD ENGINEERING\nNetwork professional services\nE2E monitoring & observability\nNetwork automation\nNetwork testing\nNetwork infrastructure design\nDATA\nData engineering\nData science\nRESEARCH AND DEVELOPMENT\nR&D\nFOR STARTUPS\nMVP software development\nSoftware development for startups\nTechnologies\nCase studies\nBlog\nKnowledge\nNetworks\nEquipment\nEnvironment\nData\nSecurity\nN\nE\nE\nD\nS\nKnowledge\nEXPERTISE\nBlog\nResources\nPUBLICATIO

In [21]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [22]:
# create_brochure("HuggingFace", "https://huggingface.co")
create_brochure("CodiLime", "https://codilime.com/")


Selecting relevant links for https://codilime.com/ by calling gpt-5-nano
Found 58 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# CodiLime Company Brochure

---

## About CodiLime

Founded in 2011, CodiLime is a premier software and network engineering company specializing in advanced technology-driven solutions. Serving semiconductor manufacturers, networking vendors, telecom companies, and software solution providers, CodiLime is dedicated to linking exceptional engineering talent with deep business domain expertise. This synergy delivers tailored, high-quality solutions with a focus on the customer's N.E.E.D.S.—Networks, Equipment, Environment, Data, and Security.

---

## Our Expertise & Services

### N.E.E.D.S. Framework

CodiLime’s core competencies revolve around five key domains:

- **Networks:** Designing, deploying, and operating future-proof networks that are traditional, virtualized, or cloud-native. Services include network infrastructure design, automation, professional network services, end-to-end monitoring, and network testing.
  
- **Equipment:** Providing software solutions built around hardware infrastructure essential to everyday operations, covered by multiple abstraction layers and APIs.

- **Environment:** Optimizing operational environments for secure, scalable, and reliable systems.

- **Data:** Offering data engineering and data science services to harness business insights and drive innovation.

- **Security:** Implementing security measures including AI-based web application firewalls and hardware TCP offloading to protect critical assets.

### Service Portfolio

- **Design & Product Development:** From product design to embedded systems.
  
- **Software Engineering:** Frontend, backend, low-level engineering, DevOps, platform engineering, and test automation.

- **Network & Cloud Engineering:** Comprehensive network services, including automation, testing, and monitoring.

- **Data & Research:** Cutting-edge R&D focused on AI/ML applications for networks, plus data science and engineering.

- **Startup Support:** MVP and full-cycle software development tailored for startups.

---

## Technology & Innovation

CodiLime is committed to innovation, highlighting expertise in:

- AI and Machine Learning applications for networking
- Cloud-native technologies such as Kubernetes
- Advanced networking protocols and hardware offloading techniques
- Research publications on SONiC (Software for Open Networking in the Cloud) and other emerging technologies

---

## Company Culture

- **Engineering Excellence:** A culture driven by high technical standards and continuous learning.
  
- **Collaboration:** Bridging engineering talent and domain knowledge to foster innovation.
  
- **Customer-Centric:** Tailoring solutions to meet exact business and technological needs.

- **Innovation Focus:** Active participation in R&D ensures staff work at the forefront of technology.

---

## Our Customers

Trusted by leaders across multiple industries, CodiLime proudly supports:

- Semiconductor manufacturers
- Networking technology vendors
- Telecommunications providers
- Software solution integrators and startups

---

## Career Opportunities

At CodiLime, careers are built on:

- Working with cutting-edge technologies in software and network engineering.
- Opportunities in product design, embedded systems, cloud engineering, data science, and R&D.
- Engaging projects for high-profile clients and startups.
- A supportive environment prioritizing skill development, innovation, and delivery excellence.

**Join CodiLime to innovate and shape the future of networking technology.**

---

## Contact Us

Explore how CodiLime can address your technology N.E.E.D.S. and enable business transformation.

Website: [CodiLime](https://codilime.com)  
Email and contact details available on the website.

---

*Let’s innovate together with CodiLime - your strategic partner in networking and software engineering.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [23]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [25]:
# stream_brochure("HuggingFace", "https://huggingface.co")
stream_brochure("CodiLime", "https://codilime.com/")

Selecting relevant links for https://codilime.com/ by calling gpt-5-nano
Found 22 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# CodiLime: Your Trusted Networking Expert & Strategic Partner

---

### Who We Are

Founded in 2011, **CodiLime** is a leading software and network engineering company specializing in delivering custom-tailored solutions to technology-driven clients worldwide. We serve semiconductor manufacturers, networking vendors, telecom operators, and software solution providers. Our mission is to link exceptional engineering talent with deep business domain expertise to ensure **delivery excellence** across all projects.

---

### Our Focus: N.E.E.D.S.

CodiLime centers its expertise around five core pillars—**Networks, Equipment, Environment, Data, and Security (N.E.E.D.S.)**—covering every aspect of modern infrastructure and software development.

---

### Our Services

#### Design & Product Innovation
- Product design with a user-centered approach

#### Software Engineering
- Frontend & Backend Development  
- Low-level Engineering  
- DevOps & Platform Engineering  
- Test Automation  
- Embedded Systems

#### Network & Cloud Engineering
- Network Professional Services  
- End-to-End Monitoring & Observability  
- Network Automation & Testing  
- Network Infrastructure Design

#### Data Services
- Data Engineering  
- Data Science

#### Research & Development
- Cutting-edge R&D projects focused on networking and software innovation

#### Startup Support
- MVP Software Development  
- Tailored Software Solutions for Startups

---

### Technology & Expertise

We leverage state-of-the-art technologies and methodologies to craft future-proof, automated networks—whether traditional, virtualized, or cloud-native. Our knowledge base includes advanced topics such as AI/ML for networks, Kubernetes application networking, hardware TCP offloading, and AI-based web application firewalls.

Explore our thought leadership through:
- Technical Publications & Case Studies  
- Expert Blog Posts  
- Industry News & Insights

---

### Our Customers

We proudly partner with industry leaders in:
- Semiconductor Manufacturing  
- Networking Vendors  
- Telecommunications  
- Software Solutions Providers

Our clients rely on us for:
- High-quality engineering expertise  
- Innovative and scalable solutions  
- Seamless integration into their existing infrastructure

---

### Company Culture

At CodiLime, innovation thrives in a collaborative environment that values technical excellence and continual learning. Our teams are empowered to explore, experiment, and push the boundaries of networking and software engineering.

We believe in:
- Building a culture of trust and transparency  
- Encouraging professional growth and knowledge sharing  
- Supporting a balanced and flexible work environment

---

### Careers at CodiLime

We are constantly looking for talented professionals passionate about:
- Software development across different layers and platforms  
- Network and cloud engineering  
- Data science and engineering  
- Research and innovation in networking technology

Join a company that values your expertise and invests in your personal and professional growth. Be part of projects that drive innovation in a fast-evolving industry.

**Find out about current job openings** on our Careers page and start your journey with CodiLime!

---

### Contact Us

Ready to innovate together?  
Visit us at www.codilime.com or reach out through our Contact page to explore how we can support your business needs.

---

CodiLime — Delivering excellence for your N.E.E.D.S.

In [26]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

# stream_brochure("HuggingFace", "https://huggingface.co")
stream_brochure("CodiLime", "https://codilime.com/")

Selecting relevant links for https://codilime.com/ by calling gpt-5-nano
Found 39 relevant links


KeyboardInterrupt: 

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>